In [1]:
# Assignment 25 - Prompting & LangChain Chains

#This assignment demonstrates Prompt Templates, Chat Prompt Templates, Structured Output using Pydantic, LangChain Chains, Runnables, and LCEL for building modular Generative AI applications.

#installing all the libraires needed for the project
!pip -q install langchain langchain-openai langchain-community langchain-core faiss-cpu pydantic pypdf tiktoken pandas openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
!pip -q install langchain-text-splitters # Temporarily added to resolve ModuleNotFoundError within this cell

import os
from getpass import getpass

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders.csv_loader import CSVLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter # Updated import path
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
print("Libraries Imported Successfully")

Libraries Imported Successfully


In [5]:
#loading all the pdf, csv and txt files
pdf_loader = PyPDFLoader("DevAnand_Resume.pdf")
pdf_docs = pdf_loader.load()
print("PDF Loaded :", len(pdf_docs))

txt_loader = TextLoader("notes.txt")
txt_docs = txt_loader.load()
print("Text Loaded :", len(txt_docs))

csv_loader = CSVLoader("employees.csv")
csv_docs = csv_loader.load()
print("CSV Loaded :", len(csv_docs))

PDF Loaded : 1
Text Loaded : 1
CSV Loaded : 5


In [6]:
#making the combination of all 3 files
documents = pdf_docs + txt_docs + csv_docs
print("Total Documents :",len(documents))

Total Documents : 7


In [7]:
# OpenAI API Setup (Using Colab Secrets)

from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
llm = ChatOpenAI(model="gpt-4.1-mini",temperature=0)
print("OpenAI Model Loaded Successfully.")

OpenAI Model Loaded Successfully.


In [ ]:
#splitting the documents and creating the embeddings
splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
docs = splitter.split_documents(documents)
print("Chunks Created :",len(docs))

embeddings = OpenAIEmbeddings()
print("Embeddings Ready")

#now creating the faiss
vectorstore = FAISS.from_documents(docs,embeddings)
retriever = vectorstore.as_retriever()
print("Vector Store Created")

In [ ]:
#Task 1 - Prompt Template
template = """You are a helpful AI assistant.
Answer the following question clearly.
Question:
{question}"""

prompt = PromptTemplate(template=template,input_variables=["question"])

chain = prompt | llm
questions = ["What is Artificial Intelligence?","Explain Machine Learning.","What is LangChain?"]
for q in questions:
    print("\nQuestion :",q)
    answer = chain.invoke({"question":q})
    print(answer.content)
    print("-"*60)

In [ ]:
## Task 2 - Chat Prompt Template
chat_prompt = ChatPromptTemplate.from_messages([("system","You are a helpful AI tutor."),("human","{question}")])
chat_chain = chat_prompt | llm
questions = ["What is NLP?","What is Generative AI?"]

for q in questions:
    print("\nQuestion :",q)
    response = chat_chain.invoke({"question":q})
    print(response.content)
    print("-"*60)

### Observation

PromptTemplate is suitable for simple prompts.

ChatPromptTemplate is better for chat conversations because it separates system and user messages.

In [ ]:
## Task 3 - Structured Output using Pydantic
class Answer(BaseModel):
    answer: str = Field(description="Answer")
    confidence: float = Field(description="Confidence Score")
    source: str = Field(description="Source")
parser = PydanticOutputParser(pydantic_object=Answer)
prompt = PromptTemplate(

template="""
Answer the question.
{format_instructions}
Question:
{question}
""",input_variables=["question"],
partial_variables={"format_instructions":parser.get_format_instructions()})
chain = prompt | llm | parser
result = chain.invoke({"question":"Explain Retrieval Augmented Generation."})
print(result)

### Observation

Pydantic ensures that the LLM always returns structured output with the required fields.

In [ ]:
## Task 4 - Validation & Error Handling
#In this task, I handled invalid LLM outputs using a simple try-except block.

try:
    result = chain.invoke({"question": "Explain LangChain."})
    print("Structured Output:")
    print(result)
except Exception as e:
    print("Invalid output received.")
    print(e)

print("\nObservation:")
print("- Pydantic validates the response format.")
print("- If the format is incorrect, an exception is raised.")

In [ ]:
## Task 5 - Simple Chain
from langchain_core.prompts import PromptTemplate

simple_prompt = PromptTemplate.from_template(
"""Answer the following question.
Question:{question}""")

simple_chain = simple_prompt | llm
response = simple_chain.invoke({"question":"What is Retrieval-Augmented Generation?"})

print(response.content)
print("\nObservation:")
print("- The chain consists of Prompt → LLM → Response.")

In [ ]:
## Task 6 - Conditional Chain
from langchain_core.runnables import RunnableBranch, RunnableLambda
retriever = vectorstore.as_retriever()
def retrieve_answer(question):
    docs = retriever.invoke(question)

    context = "\n".join([doc.page_content for doc in docs])
    prompt = f"""
Use the following context to answer the question
Context:{context}
Question:{question}"""

    return llm.invoke(prompt).content
def direct_answer(question):
    return llm.invoke(question).content

conditional_chain = RunnableBranch((
lambda x: any(word in x.lower() for word in ["who","what","when","where","pdf","employee","resume"]),
RunnableLambda(retrieve_answer)),RunnableLambda(direct_answer))
questions = ["What skills are mentioned in the resume?","Tell me a joke."]

for q in questions:
    print("\nQuestion:",q)
    answer = conditional_chain.invoke(q)
    print(answer)
    print("-"*60)
print("\nObservation:")
print("- Factual questions use document retrieval.")
print("- General questions use the LLM directly.")

In [ ]:
## Task 7 - Parallel Chain
from langchain_core.runnables import RunnableParallel
from langchain_core.runnables import RunnableLambda

def generate_answer(question):
    return llm.invoke(question).content

def generate_summary(question):
    prompt = f"Summarize this question briefly:\n{question}"
    return llm.invoke(prompt).content

def generate_followup(question):
    prompt = f"Generate two follow-up questions for:\n{question}"
    return llm.invoke(prompt).content

parallel_chain = RunnableParallel(
answer=RunnableLambda(generate_answer),
summary=RunnableLambda(generate_summary),
follow_up=RunnableLambda(generate_followup))

result = parallel_chain.invoke("What is Artificial Intelligence?")
print("Answer:\n")
print(result["answer"])

print("\nSummary:\n")
print(result["summary"])

print("\nFollow-up Questions:\n")
print(result["follow_up"])

print("\nObservation:")
print("- Multiple chains were executed in parallel.")
print("- Each chain performs a different task.")

## Task 8 - Runnable Basics

In this task, I converted a prompt into a runnable pipeline using LCEL and RunnablePassthrough.

In [ ]:
#task8
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate.from_template("""Answer the following question
Question:{question}""")

runnable_chain = (RunnablePassthrough()| prompt| llm| StrOutputParser())
response = runnable_chain.invoke({"question": "What is LangChain?"})

print(response)
print("\nObservation:")
print("- RunnablePassthrough forwards the input.")
print("- LCEL uses the | operator to connect components.")

In [ ]:
## Task 9 - LCEL Based RAG Chain
from langchain_core.output_parsers import StrOutputParser

rag_prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.
Answer only using the given context
Context:{context}
Question:{question}""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = ({"context": retriever | format_docs,"question": RunnablePassthrough()}| rag_prompt| llm| StrOutputParser())
questions = ["What skills are mentioned in the resume?","What is the employee salary?","Summarize the notes."]

for q in questions:
    print("\nQuestion:")
    print(q)
    answer = rag_chain.invoke(q)
    print("\nAnswer:")
    print(answer)
    print("-" * 60)

print("\nObservation:")
print("- Documents were retrieved from FAISS.")
print("- LCEL connected Retriever, Prompt, LLM and Output Parser.")

# Task 10 – Observations & Insights

### 1. Why is structured output important?

Structured output helps keep AI responses organized and consistent. Since the output follows a predefined format, it becomes easier to read, validate, and integrate with applications or databases

### 2. Advantages of LCEL over traditional chains

LCEL makes it much easier to build and manage LLM workflows. Its syntax is simple and easy to understand, making pipelines more readable.

### 3. When should we use Parallel Chains and Conditional Chains?

Parallel Chains are best used when multiple independent tasks can run simultaneously

Conditional Chains are useful when the workflow needs to change based on the user's request